# PGM end-to-end pipeline

This notebook runs **every stage** of the project in order: load raw data, preprocess, optionally explore QC plots, cluster cells, fit three Gaussian graphical model (GGM) variants, evaluate them, and record a reproducibility snapshot.

**Model sketch:** Graphical Lasso minimizes \(-\log\det\Theta + \mathrm{tr}(S\Theta) + \alpha\|\Theta\|_1\) with \(\Theta \succeq 0\). For mixture states, a weighted scatter \(S^{(k)}\) is used; the KG-biased stage blends covariance with a prior \(S' = S^{(k)} + \lambda P\) (see `src/pgm/models/`).

Artifacts land under `data/`, `reports/`, and `logs/` as configured in `configs/default.yaml` (and `smoke.yaml` when smoke mode is on).


## 1. Setup

The package expects to run with the **repository root** on `sys.path` and as the working directory so paths like `configs/`, `data/`, and `datasets/` resolve correctly. This cell moves to the parent of `notebooks/` and imports the pipeline entrypoints.

Use `SMOKE=1` in the environment or pass `smoke=True` to `load_config` for faster, smaller runs (merged overlay from `configs/smoke.yaml`).


In [ ]:
from pathlib import Path
import json
import os
import time

os.chdir(Path("..").resolve())

FORCE_RERUN = True  # set True to ignore existing checkpoints / outputs where supported

from pgm.config.loader import load_config
from pgm.pipelines.ingest import ingest_pbmc_pipeline
from pgm.pipelines.preprocess import run_preprocess_pipeline
from pgm.pipelines.eda import run_eda
from pgm.pipelines.cluster import run_clustering_pipeline
from pgm.pipelines.global_ggm import run_global_ggm_pipeline
from pgm.pipelines.soft_ggm import run_soft_weighted_ggm_pipeline
from pgm.pipelines.kg_pipeline import run_kg_pipeline
from pgm.pipelines.evaluation import run_evaluation_pipeline
from pgm.utils.env import snapshot_run_environment


## 2. Configuration

Loads `configs/default.yaml`, merges `configs/smoke.yaml` when smoke mode is enabled, and validates into a `ProjectConfig` (paths, QC thresholds, GGM hyperparameters, STRING/KG settings, evaluation bootstrap sizes). All later stages read this single object so behavior is consistent.


In [2]:
cfg = load_config(smoke=False)
cfg


ProjectConfig(paths=PathsConfig(project_root=None), data=DataPathsConfig(tenx_mtx_dir=WindowsPath('datasets/filtered_gene_bc_matrices/hg19'), interim_dir=WindowsPath('data/interim'), processed_dir=WindowsPath('data/processed'), checkpoint_dir=WindowsPath('data/checkpoints'), external_dir=WindowsPath('data/external'), string_cache_dir=WindowsPath('data/external/string_cache')), reports=ReportsPathsConfig(figures_dir=WindowsPath('reports/figures'), eda_dir=WindowsPath('reports/eda'), results_dir=WindowsPath('reports/results')), run=RunConfig(random_seed=42), logging=LoggingConfig(level='INFO', console=True, log_dir=WindowsPath('logs'), rotating_max_bytes=10485760, rotating_backup_count=3), checkpoint=CheckpointRuntimeConfig(subdirectory=''), eda=EdaRuntimeConfig(n_cells_profiling_hint=None, umap_neighbors=15, umap_min_dist=0.5), ingestion=IngestionConfig(dataset_name='pbmc3k', write_interim_filename='pbmc3k_raw.h5ad'), preprocessing=PreprocessingConfig(target_sum=10000.0, min_genes=200, 

## 3. Data ingestion

Reads the 10x PBMC matrix from `data.tenx_mtx_dir`, performs basic validation, writes a JSON summary under `reports/results/`, and saves interim AnnData to `data/interim/` (`ingestion.write_interim_filename`). Checkpoints avoid recomputation when `FORCE_RERUN` is False.


In [3]:
t_ingest = time.perf_counter()
interim_raw_h5ad = ingest_pbmc_pipeline(cfg, force=FORCE_RERUN)
print(f"Ingestion → {interim_raw_h5ad} ({time.perf_counter() - t_ingest:.2f}s)")


Ingestion → F:\Research\PGM\data\interim\pbmc3k_raw.h5ad (0.37s)


## 4. Preprocessing

Standard single-cell workflow: QC filters (genes/cells, mitochondrial fraction), normalization to a target sum, log transform, highly variable genes, scaling, PCA. Writes `data/processed/` AnnData including `X_pca` for downstream neighbors and mixture models.


In [4]:
t_pre = time.perf_counter()
processed_h5ad = run_preprocess_pipeline(cfg, interim_raw_h5ad, force=FORCE_RERUN)
print(f"Preprocess → {processed_h5ad} ({time.perf_counter() - t_pre:.2f}s)")


[05/13/26 05:25:35] INFO     Running PCA n_comps=50 (nhvg=2000)                                      ]8;id=16682981;file://F:\Research\PGM\src\pgm\preprocessing\pipeline.py\pipeline.py]8;;\:]8;id=16682982;file://F:\Research\PGM\src\pgm\preprocessing\pipeline.py#79\79]8;;\

[05/13/26 05:25:36] INFO     Preprocessed shape=(2698, 2000); HVG count=2000                         ]8;id=16682988;file://F:\Research\PGM\src\pgm\preprocessing\pipeline.py\pipeline.py]8;;\:]8;id=16682989;file://F:\Research\PGM\src\pgm\preprocessing\pipeline.py#81\81]8;;\

                    INFO     Saved AnnData checkpoint                                              ]8;id=16682996;file://F:\Research\PGM\src\pgm\utils\checkpoint.py\checkpoint.py]8;;\:]8;id=16682997;file://F:\Research\PGM\src\pgm\utils\checkpoint.py#84\84]8;;\
                             F:\Research\PGM\data\checkpoints\preprocess_done.h5ad (AnnData)                       

                    INFO     Preprocessed 7.06s →                                                  ]8;id=16683004;file://F:\Research\PGM\src\pgm\pipelines\preprocess.py\preprocess.py]8;;\:]8;id=16683005;file://F:\Research\PGM\src\pgm\pipelines\preprocess.py#52\52]8;;\
                             F:\Research\PGM\data\processed\pbmc3k_processed.h5ad shape=(2698,                     
                             2000)                                                                                 

Preprocess → F:\Research\PGM\data\processed\pbmc3k_processed.h5ad (7.08s)


### 4b. Processed matrix sanity check

Loads the written processed `.h5ad` and prints non-finite counts, value range, and how many genes have ~zero standard deviation across cells. NaNs after scaling usually mean zero-variance genes; Graphical Lasso is sensitive to singular covariance. Preprocessing now drops those genes before PCA.

In [ ]:
import scanpy as sc

from pgm.utils.adata_audit import format_matrix_summary, summarize_adata_matrix

_qc = sc.read_h5ad(processed_h5ad)
print(format_matrix_summary(summarize_adata_matrix(_qc)))
print("\nobs columns (sample):", list(_qc.obs.columns[:12]))
_qc.obs.head()

## 5. Exploratory data analysis (optional)

Builds QC-style figures under `reports/figures/eda/` and a short markdown summary under `reports/eda/`. This stage is **not** required for the modeling chain but helps verify preprocessing; it uses the processed AnnData path and respects smoke sampling hints from config.


In [5]:
t_eda = time.perf_counter()
eda_summary_path = run_eda(cfg, processed_h5ad, force=FORCE_RERUN)
print(f"EDA summary → {eda_summary_path} ({time.perf_counter() - t_eda:.2f}s)")


F:\Research\PGM\src\pgm\pipelines\eda.py:83: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(adata, target_sum=1e4)
f:\Research\PGM\.venv\Lib\site-packages\scanpy\preprocessing\_simple.py:377: RuntimeWarning: invalid value encountered in log1p
  np.log1p(x, out=x)


ValueError: Input X contains NaN.
PCA does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 6. Clustering and latent visualization

Computes nearest neighbors on PCA, Leiden clustering, UMAP embeddings, and fits a Gaussian mixture on the configured PCA representation to obtain **soft** component assignments. Saves clustered AnnData (default sibling `*_clustered.h5ad` in `data/processed/`) and Leiden/GMM diagnostic figures under `reports/figures/clustering/`.


In [6]:
t_cl = time.perf_counter()
clustered_h5ad = run_clustering_pipeline(cfg, processed_h5ad, force=FORCE_RERUN)
print(f"Clustering → {clustered_h5ad} ({time.perf_counter() - t_cl:.2f}s)")


f:\Research\PGM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[05/13/26 05:26:25] INFO     GMM fit done; mean soft assignment entropy 0.0414                  ]8;id=16683012;file://F:\Research\PGM\src\pgm\clustering\heterogeneity.py\heterogeneity.py]8;;\:]8;id=16683013;file://F:\Research\PGM\src\pgm\clustering\heterogeneity.py#72\72]8;;\

[05/13/26 05:26:26] INFO     Saved AnnData checkpoint                                              ]8;id=16683018;file://F:\Research\PGM\src\pgm\utils\checkpoint.py\checkpoint.py]8;;\:]8;id=16683019;file://F:\Research\PGM\src\pgm\utils\checkpoint.py#84\84]8;;\
                             F:\Research\PGM\data\checkpoints\cluster_done.h5ad (AnnData)                          

                    INFO     Clustering done 23.62s →                                                 ]8;id=16683026;file://F:\Research\PGM\src\pgm\pipelines\cluster.py\cluster.py]8;;\:]8;id=16683027;file://F:\Research\PGM\src\pgm\pipelines\cluster.py#59\59]8;;\
                             F:\Research\PGM\data\processed\pbmc3k_processed_clustered.h5ad                        
                             (n_obs=2698)                                                                          

Clustering → F:\Research\PGM\data\processed\pbmc3k_processed_clustered.h5ad (23.62s)


## 7. Global graphical Lasso (one graph for all cells)

Fits a **single** sparse inverse covariance (precision) matrix and adjacency from the full processed gene set (subject to model code paths). Serialized bundle under `data/checkpoints/global_ggm.joblib` for evaluation and comparison.


In [7]:
t_gg = time.perf_counter()
global_ggm_joblib = run_global_ggm_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"Global GGM → {global_ggm_joblib} ({time.perf_counter() - t_gg:.2f}s)")


[05/13/26 05:26:36] INFO     Global GGM fitting on X shape (2698, 2000) alpha=None                 ]8;id=16683034;file://F:\Research\PGM\src\pgm\models\global_ggm.py\global_ggm.py]8;;\:]8;id=16683035;file://F:\Research\PGM\src\pgm\models\global_ggm.py#22\22]8;;\

f:\Research\PGM\.venv\Lib\site-packages\sklearn\covariance\_graph_lasso.py:171: RuntimeWarning: divide by zero encountered in scalar divide
  precision_[idx, idx] = 1.0 / (
f:\Research\PGM\.venv\Lib\site-packages\sklearn\covariance\_graph_lasso.py:175: RuntimeWarning: invalid value encountered in multiply
  precision_[indices != idx, idx] = -precision_[idx, idx] * coefs
f:\Research\PGM\.venv\Lib\site-packages\sklearn\covariance\_graph_lasso.py:176: RuntimeWarning: invalid value encountered in multiply
  precision_[idx, indices != idx] = -precision_[idx, idx] * coefs
f:\Research\PGM\.venv\Lib\site-packages\sklearn\covariance\_graph_lasso.py:171: RuntimeWarning: divide by zero encountered in scalar divide
  precision_[idx, idx] = 1.0 / (
f:\Research\PGM\.venv\Lib\site-packages\sklearn\covariance\_graph_lasso.py:175: RuntimeWarning: invalid value encountered in multiply
  precision_[indices != idx, idx] = -precision_[idx, idx] * coefs
f:\Research\PGM\.venv\Lib\site-packages\sklearn\covari

FloatingPointError: Non SPD result: the system is too ill-conditioned for this solver. The system is too ill-conditioned for this solver

## 8. Soft mixture–weighted GGM (per latent state)

Uses GMM soft labels to fit **one graphical model per mixture component**, weighting cells by responsibility. Saves `data/checkpoints/soft_weighted_ggm.joblib` and component-level sparsity figures under `reports/figures/ggm/`. Compares heterogeneous structure across states.


In [ ]:
t_soft = time.perf_counter()
soft_ggm_joblib = run_soft_weighted_ggm_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"Soft GGM → {soft_ggm_joblib} ({time.perf_counter() - t_soft:.2f}s)")


## 9. Knowledge-graph prior GGM (STRING)

Queries STRING (cached under `data/external/string_cache/`) for high-confidence edges among HVGs, builds a sparse prior matrix \(P\), and fits KG-biased GGMs per mixture state. Outputs `data/checkpoints/kg_soft_ggm.joblib` and related exports for evaluation against STRING-derived reference pairs.


In [ ]:
t_kg = time.perf_counter()
kg_joblib = run_kg_pipeline(cfg, clustered_h5ad, force=FORCE_RERUN)
print(f"KG GGM → {kg_joblib} ({time.perf_counter() - t_kg:.2f}s)")


## 10. Evaluation

Aggregates **bootstrap stability** of the global adjacency, **STRING precision/recall** for the global graph, **degree/sparsity** summaries, and **cross-state similarity** (Jaccard / Frobenius) for soft and KG adjacencies where applicable. Writes `reports/results/metrics_summary.csv`.


In [ ]:
t_ev = time.perf_counter()
metrics_csv = run_evaluation_pipeline(
    cfg,
    clustered_h5ad,
    global_ggm_joblib,
    soft_ggm_joblib,
    kg_joblib,
    force=FORCE_RERUN,
)
print(f"Metrics → {metrics_csv} ({time.perf_counter() - t_ev:.2f}s)")


## 11. Run environment snapshot

Persists dependency versions and run metadata for reproducibility (location defined by `pgm.utils.env.snapshot_run_environment` and config paths). Use this alongside saved checkpoints for paper methods and debugging.


In [ ]:
t_env = time.perf_counter()
env_json = snapshot_run_environment(cfg, tag="full_pipeline")
print(f"Environment → {env_json} ({time.perf_counter() - t_env:.2f}s)")


## 12. Summary and exports

Collects all stage paths in one dict (same keys as `run_full_pipeline` in `src/pgm/pipelines/full_pipeline.py`), total wall time, and optionally writes a JSON summary under `reports/results/`.


In [ ]:
summary = {
    "interim_raw_h5ad": interim_raw_h5ad,
    "processed_h5ad": processed_h5ad,
    "eda_summary": eda_summary_path,
    "clustered_h5ad": clustered_h5ad,
    "global_ggm_joblib": global_ggm_joblib,
    "soft_ggm_joblib": soft_ggm_joblib,
    "kg_joblib": kg_joblib,
    "metrics_csv": metrics_csv,
    "env_json": env_json,
}
summary["runtime_s"] = time.perf_counter() - t_ingest

out_json = Path("reports/results/final_pipeline_summary.json")
out_json.parent.mkdir(parents=True, exist_ok=True)
out_json.write_text(
    json.dumps({k: str(v) for k, v in summary.items()}, indent=2),
    encoding="utf-8",
)
print(f"Wrote {out_json}")
summary
